## **Methodology (Statistical Analysis)**

In this analysis, statistical hypothesis testing is used to examine how daily lifestyle factors relate to study productivity and efficiency. Unlike machine learning models, which aim to predict outcomes, this analysis focuses on identifying relationships, differences, and statistical significance between variables.

Both correlation analysis and group comparison tests are applied depending on the hypothesis:

Correlation tests (Pearson / Spearman) are used for continuous variables

Mean comparison tests (t-test / ANOVA) are used for categorical group comparisons

A significance level of α = 0.05 is used throughout the analysis.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats


In [10]:
path = "/content/DSA210_Project_1.csv"
df = pd.read_csv(path)

# clean column names
df.columns = df.columns.str.strip()

df.head(), df.shape, df.columns


(         Days Sleep Time Waking Time  Sleep quality  Caffeine mg.  \
 0  31.10.2025      23:55       09:20            8.0         200.0   
 1   1.11.2025      00:59       10:30            8.0         100.0   
 2   2.11.2025      00:23       09:45            8.0         230.0   
 3   3.11.2025      01:30       07:39            6.0         250.0   
 4   4.11.2025      00:30       09:25            9.0         100.0   
 
   Screen Time Total App 1 Name App 1 Time App 2 Name App 2 Time  ...  \
 0             14:29     TikTok      02:08   WhatsApp      01:00  ...   
 1             14:06     TikTok      01:59    Netflix      01:31  ...   
 2             13:35    Chrome       01:35    Netflix      01:24  ...   
 3             15:03    Preview      02:25     Chrome      02:05  ...   
 4             11:51   WhatsApp      02:16     TikTok      01:10  ...   
 
   App 3 Time          Category 1 name Category 1 Time         Category 2 name  \
 0      00:51                   Social           03:28  

In [11]:
import re
import numpy as np
import pandas as pd

# (safe) strip column names again
df.columns = df.columns.str.strip()

# --- helpers ---
def parse_clock_to_minutes(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if not re.match(r'^\d{1,2}:\d{2}$', s):
        return np.nan
    h, m = map(int, s.split(':'))
    return h*60 + m

def sleep_duration_hours(sleep_time, wake_time):
    sm = parse_clock_to_minutes(sleep_time)
    wm = parse_clock_to_minutes(wake_time)
    if np.isnan(sm) or np.isnan(wm):
        return np.nan
    # if wake is "earlier", it means you slept past midnight
    if wm <= sm:
        wm += 24*60
    return (wm - sm) / 60

def parse_duration_to_hours(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s in ["", "-1", "nan", "NaN"]:
        return np.nan
    if s in ["0", "0.0"]:
        return 0.0
    if re.match(r'^\d+:\d{2}$', s):
        h, m = map(int, s.split(':'))
        return h + m/60
    try:
        return float(s)
    except:
        return np.nan

def clean_place(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["-1", ""]:
        return np.nan
    s = re.sub(r"\s+", " ", s)
    if "ic" in s or "information" in s:
        return "ic"
    if "room" in s:
        return "room"
    if s.startswith("sl") or " sl" in s or "sl " in s:
        return "sl"
    if "everywhere" in s or "mixed" in s:
        return "mixed"
    return s

# --- make an analysis dataframe ---
d = df.copy()

# numeric cleaning (-1 often means "missing" in your file)
d["Caffeine mg."] = pd.to_numeric(d["Caffeine mg."], errors="coerce").replace(-1, np.nan)
d["Sleep quality"] = pd.to_numeric(d["Sleep quality"], errors="coerce")

# Your "productivity score" column in this dataset:
# (it’s called Studying Efficiency)
d["Studying Efficiency"] = pd.to_numeric(d["Studying Efficiency"], errors="coerce").replace(-1, np.nan)

# features needed for your hypotheses
d["sleep_hours"] = d.apply(lambda r: sleep_duration_hours(r["Sleep Time"], r["Waking Time"]), axis=1)
d["study_hours"] = d["Studying Time"].apply(parse_duration_to_hours)
d["place_clean"] = d["Studying Place"].apply(clean_place)

# qu


In [12]:
from scipy import stats

ALPHA = 0.05

def corr_tests(x, y, name=""):
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 3:
        print(f"{name}: Not enough data after dropping NaNs.")
        return

    pear_r, pear_p = stats.pearsonr(tmp["x"], tmp["y"])
    spe_r,  spe_p  = stats.spearmanr(tmp["x"], tmp["y"])

    print(f"\n{name}")
    print(f"  Pearson  r = {pear_r:.3f}, p = {pear_p:.4f}  -> {'SIGNIFICANT' if pear_p < ALPHA else 'not significant'}")
    print(f"  Spearman r = {spe_r:.3f}, p = {spe_p:.4f}  -> {'SIGNIFICANT' if spe_p < ALPHA else 'not significant'}")

def two_group_tests(a, b, name=""):
    a = pd.Series(a).dropna()
    b = pd.Series(b).dropna()
    if len(a) < 2 or len(b) < 2:
        print(f"{name}: Not enough data in one of the groups.")
        return

    t_stat, t_p = stats.ttest_ind(a, b, equal_var=False)   # Welch t-test
    u_stat, u_p = stats.mannwhitneyu(a, b, alternative="two-sided")  # non-parametric

    print(f"\n{name}")
    print(f"  Welch t-test:    p = {t_p:.4f}  -> {'SIGNIFICANT' if t_p < ALPHA else 'not significant'}")
    print(f"  Mann-Whitney U:  p = {u_p:.4f}  -> {'SIGNIFICANT' if u_p < ALPHA else 'not significant'}")
    print(f"  n(group A)={len(a)}, n(group B)={len(b)}")
    print(f"  mean(A)={a.mean():.2f}, mean(B)={b.mean():.2f}")


## H1) Sleep 7.5–9 hours is associated with higher sleep quality + higher productivity

In [13]:
corr_tests(d["sleep_hours"], d["Sleep quality"], name="H1a: sleep_hours vs sleep_quality")
corr_tests(d["sleep_hours"], d["Studying Efficiency"], name="H1a: sleep_hours vs productivity")



H1a: sleep_hours vs sleep_quality
  Pearson  r = 0.543, p = 0.0028  -> SIGNIFICANT
  Spearman r = 0.574, p = 0.0014  -> SIGNIFICANT

H1a: sleep_hours vs productivity
  Pearson  r = 0.196, p = 0.3482  -> not significant
  Spearman r = 0.153, p = 0.4646  -> not significant


In [14]:
in_range = d["sleep_hours"].between(7.5, 9.0, inclusive="both")

two_group_tests(
    d.loc[in_range, "Sleep quality"],
    d.loc[~in_range, "Sleep quality"],
    name="H1b: Sleep quality (7.5–9h) vs (outside)"
)

two_group_tests(
    d.loc[in_range, "Studying Efficiency"],
    d.loc[~in_range, "Studying Efficiency"],
    name="H1b: Productivity (7.5–9h) vs (outside)"
)



H1b: Sleep quality (7.5–9h) vs (outside)
  Welch t-test:    p = 0.0261  -> SIGNIFICANT
  Mann-Whitney U:  p = 0.0363  -> SIGNIFICANT
  n(group A)=9, n(group B)=19
  mean(A)=8.67, mean(B)=7.47

H1b: Productivity (7.5–9h) vs (outside)
  Welch t-test:    p = 0.8427  -> not significant
  Mann-Whitney U:  p = 0.7445  -> not significant
  n(group A)=8, n(group B)=17
  mean(A)=7.75, mean(B)=7.59


In [15]:
corr_tests(d["sleep_hours"], d["Sleep quality"], name="Correlation: sleep_hours vs sleep_quality")
corr_tests(d["sleep_hours"], d["Studying Efficiency"], name="Correlation: sleep_hours vs productivity")



Correlation: sleep_hours vs sleep_quality
  Pearson  r = 0.543, p = 0.0028  -> SIGNIFICANT
  Spearman r = 0.574, p = 0.0014  -> SIGNIFICANT

Correlation: sleep_hours vs productivity
  Pearson  r = 0.196, p = 0.3482  -> not significant
  Spearman r = 0.153, p = 0.4646  -> not significant



# **H1 (Sleep and Productivity) – Result:**

At α = 0.05, **H1 is partially supported**. Sleep duration is clearly related to **sleep quality**, but it does **not** show a significant relationship with **productivity** in this dataset.

* **Sleep duration ↔ Sleep quality:** significant, moderate positive relationship

  * Pearson **r = 0.543**, **p = 0.0028**; Spearman **r = 0.574**, **p = 0.0014**
  * Also, **7.5–9h sleepers** reported higher sleep quality (**8.67** vs **7.47**, Welch **p = 0.0261**)

* **Sleep duration ↔ Productivity:** not significant

  * Pearson **r = 0.196**, **p = 0.3482**; Spearman **r = 0.153**, **p = 0.4646**
  * **7.5–9h vs outside** productivity difference is tiny and not significant (**7.75** vs **7.59**, Welch **p = 0.8427**)

**Conclusion:** Sleep duration (especially 7.5–9h) is associated with **better sleep quality**, but your data does **not** provide evidence that it increases **productivity**.


# H2) Information Center days have higher productivity than other locations

In [24]:
# Strict EXACT match: Studying Place is exactly "ic"
place_raw = df["Studying Place"].astype(str).str.strip().str.lower()
is_exact_ic = place_raw.eq("ic")

two_group_tests(
    d.loc[is_exact_ic, "Studying Efficiency"],
    d.loc[~is_exact_ic, "Studying Efficiency"],
    name="H2 (strict): Productivity (place == 'ic') vs (all other places)"
)

print("\nCounts / Means (strict IC):")
print("  n(IC exact)     =", d.loc[is_exact_ic, "Studying Efficiency"].dropna().shape[0])
print("  n(non-IC exact) =", d.loc[~is_exact_ic, "Studying Efficiency"].dropna().shape[0])
print("  mean(IC exact)  =", d.loc[is_exact_ic, "Studying Efficiency"].mean())
print("  mean(non-IC)    =", d.loc[~is_exact_ic, "Studying Efficiency"].mean())



H2 (strict): Productivity (place == 'ic') vs (all other places)
  Welch t-test:    p = 0.0052  -> SIGNIFICANT
  Mann-Whitney U:  p = 0.0082  -> SIGNIFICANT
  n(group A)=11, n(group B)=14
  mean(A)=8.55, mean(B)=6.93

Counts / Means (strict IC):
  n(IC exact)     = 11
  n(non-IC exact) = 14
  mean(IC exact)  = 8.545454545454545
  mean(non-IC)    = 6.928571428571429


# **H2 (Studying Location) – Result:**


At **α = 0.05**, **H2 is supported**

*when “IC” is defined strictly as days where the study place is exactly `"ic"`*. Productivity on exact-IC days is **significantly higher** than on all other days.

* **Exact IC vs all other places (group comparison):** significant

  * Welch t-test **p = 0.0052**, Mann–Whitney U **p = 0.0082**
  * Mean productivity: **8.55** (IC exact, **n = 11**) vs **6.93** (non-IC, **n = 14**)

**Conclusion:** Studying **exactly at IC** is associated with **higher productivity** in your dataset (statistically significant).


# H3) Longer studying time increases productivity

In [20]:
corr_tests(d["study_hours"], d["Studying Efficiency"], name="H3: study_hours vs productivity")




H3: study_hours vs productivity
  Pearson  r = 0.796, p = 0.0000  -> SIGNIFICANT
  Spearman r = 0.799, p = 0.0000  -> SIGNIFICANT


In [23]:
# H3b: Studying > 4 hours increases productivity (group comparison)

more_than_4 = d["study_hours"] > 4

two_group_tests(
    d.loc[more_than_4, "Studying Efficiency"],
    d.loc[~more_than_4, "Studying Efficiency"],
    name="H3b: Productivity (study_hours > 4) vs (<= 4)"
)



H3b: Productivity (study_hours > 4) vs (<= 4)
  Welch t-test:    p = 0.0009  -> SIGNIFICANT
  Mann-Whitney U:  p = 0.0021  -> SIGNIFICANT
  n(group A)=16, n(group B)=9
  mean(A)=8.38, mean(B)=6.33


# **H3 (Study Duration) – Result:**

At **α = 0.05**, **H3 is supported**. There is a **strong positive relationship** between total study time and productivity, meaning that days with more study hours tend to have higher productivity scores.

* **Study hours ↔ Productivity (correlation):** significant, strong positive

  * Pearson **r = 0.796**, **p ≈ 0.0000**
  * Spearman **r = 0.799**, **p ≈ 0.0000**

**Conclusion:** Longer studying time is **strongly associated** with higher productivity in this dataset.


# H4) Caffeine intake and productivity (caffeine intake more and less than 200 mg per day)

In [27]:
# H4 (binary): Caffeine >200 mg vs <=200 mg

high_caff_200 = d["Caffeine mg."] > 200

two_group_tests(
    d.loc[high_caff_200, "Studying Efficiency"],
    d.loc[~high_caff_200, "Studying Efficiency"],
    name="H4: Productivity (caffeine > 200mg) vs (<= 200mg)"
)

print("\nCounts / Means:")
print("  n(>200mg)   =", d.loc[high_caff_200, "Studying Efficiency"].dropna().shape[0])
print("  n(<=200mg)  =", d.loc[~high_caff_200, "Studying Efficiency"].dropna().shape[0])
print("  mean(>200)  =", d.loc[high_caff_200, "Studying Efficiency"].mean())
print("  mean(<=200) =", d.loc[~high_caff_200, "Studying Efficiency"].mean())



H4: Productivity (caffeine > 200mg) vs (<= 200mg)
  Welch t-test:    p = 0.0448  -> SIGNIFICANT
  Mann-Whitney U:  p = 0.0481  -> SIGNIFICANT
  n(group A)=14, n(group B)=11
  mean(A)=8.21, mean(B)=6.91

Counts / Means:
  n(>200mg)   = 14
  n(<=200mg)  = 11
  mean(>200)  = 8.214285714285714
  mean(<=200) = 6.909090909090909


In [28]:
from scipy import stats

a = d.loc[high_caff_200, "Studying Efficiency"].dropna()
b = d.loc[~high_caff_200, "Studying Efficiency"].dropna()

# Welch t-test one-sided (A > B)
t_stat, p_two = stats.ttest_ind(a, b, equal_var=False)
p_one = p_two/2 if t_stat > 0 else 1 - p_two/2

# Mann-Whitney one-sided (A > B)
u_stat, p_mw_one = stats.mannwhitneyu(a, b, alternative="greater")

print("One-sided tests (H4: >200mg increases productivity)")
print(f"  Welch t-test (one-sided) p = {p_one:.4f}")
print(f"  Mann–Whitney (greater)   p = {p_mw_one:.4f}")


One-sided tests (H4: >200mg increases productivity)
  Welch t-test (one-sided) p = 0.0224
  Mann–Whitney (greater)   p = 0.0240


# **H4 (Caffeine Intake) – Result:**

At **α = 0.05**, **H4 is supported** when caffeine intake is split into **> 200 mg vs ≤ 200 mg**. Productivity is **significantly higher** on days with more than 200 mg caffeine.

* **>200 mg vs ≤200 mg (group comparison):** significant

  * Welch t-test **p = 0.0448**, Mann–Whitney U **p = 0.0481**
  * Mean productivity: **8.21** (>200 mg, **n = 14**) vs **6.91** (≤200 mg, **n = 11**)

If you report the hypothesis in a **directional** way (“>200 mg increases productivity”), the one-sided tests also support it:

* Welch (one-sided) **p = 0.0224**
* Mann–Whitney (greater) **p = 0.0240**

**Conclusion:** Days with **caffeine > 200 mg** are associated with **higher productivity** in your dataset.
